In [1]:
import os, json, rich, unicodedata
import spacy
from spacy.matcher import PhraseMatcher
from skillNer.general_params import SKILL_DB
from skillNer.skill_extractor_class import SkillExtractor
from skillNer.text_class import Text, Word

# nlp = spacy.load('spacy_outputs/model-best')
nlp = spacy.load('en_core_web_lg')
skill_extractor = SkillExtractor(nlp, SKILL_DB, PhraseMatcher)

loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...


#### skillNer
an opensource skill database and extractor; used to preannotate training data more reliably

In [2]:
with open('DATA/resumes.json','r') as f:
    data = json.loads(f.read())

In [ ]:
documents = []
i = 0
for d in data:
    try:
        # ascii decode necessary to avoid index errors caused by unhandled spacy updates in skillNer
        text = unicodedata.normalize('NFKD', d ).encode('ascii', 'ignore').decode("utf-8")
        entities = []

        doc = Text(text,nlp)
        doc_text = unicodedata.normalize('NFKD', doc.transformed_text).encode('ascii', 'ignore').decode("utf-8")
        annotations = skill_extractor.annotate(doc_text)

        for n in annotations['results']['full_matches']:
            value = n['doc_node_value']
            label = SKILL_DB[n["skill_id"]]["skill_type"]
            start = doc_text.find(value)
            if start == -1:
                continue
            else:
                end = start + len(value)
                entities.append([start,end,label,value])


        for m in annotations['results']['ngram_scored']:
            value = m['doc_node_value']
            start = m['doc_node_id'][0]
            end = start + len(m['doc_node_value'])
            label = SKILL_DB[m["skill_id"]]["skill_type"]
            entities.append([start,end,label,value])

        documents.append([doc_text,entities])

    except IndexError:
        with open(f'DATA/docs_{i}.json', 'x') as j:
            print('invalid document text')
            i += 1
            j.write(json.dumps(documents))

In [ ]:
print(len(documents))